In [ ]:
import re
def is_variable(expr):
  return isinstance(expr,str) and expr.startswith('?')
def parse_expression(expr_str):
  expr_str=expr_str.replace(" ","")
  tokens = re.findall(r'\?[a-zA-Z0-9]+|[a-zA-Z0-9]+|[\(\),]', expr_str)
  def parse_tokens(token_list):
    if not token_list:
      return None
    token=token_list.pop(0)
    if token_list and token_list[0]=='(':
      token_list.pop(0)
      args=[]
      while token_list and token_list!=')':
        arg=parse_tokens(token_list)
        if arg:
          args.append(arg)
        if token_list and token_list[0]==',':
          token_list.pop(0)
      if token_list :
        token_list.pop(0)
      return [token]+args
    else:
      return token

  return parse_tokens(tokens)

def apply_substitution(sub,expr):
  if isinstance(expr, str):
        return sub.get(expr, expr)
  if isinstance(expr, list):
        return [expr[0]] + [apply_substitution(sub, arg) for arg in expr[1:]]
  return expr
def occur_check(var, expr):
  if var == expr:
    return True
  if isinstance(expr, list):
        return any(occur_check(var, arg) for arg in expr[1:])
  return False
def unify(expr1, expr2, theta=None):
    if theta is None:
        theta = {}

    if expr1 == expr2:
        return theta
    if is_variable(expr1):
        return unify_variable(expr1, expr2, theta)
    if is_variable(expr2):
        return unify_variable(expr2, expr1, theta)
    if isinstance(expr1, list) and isinstance(expr2, list):
        if expr1[0] != expr2[0] or len(expr1) != len(expr2):
            return None
        for arg1, arg2 in zip(expr1[1:], expr2[1:]):
            theta = unify(apply_substitution(theta, arg1), apply_substitution(theta, arg2), theta)
            if theta is None:
                return None
        return theta

    return None

def unify_variable(var, expr, theta):
    if var in theta:
        return unify(theta[var], expr, theta)
    if is_variable(expr) and expr in theta:
        return unify(var, theta[expr], theta)
    if occur_check(var, expr):
        return None
    new_theta = theta.copy()
    new_theta[var] = expr
    for k, v in new_theta.items():
        if k != var:
            new_theta[k] = apply_substitution({var: expr}, v)
    return new_theta

def format_expr(expr):
    if isinstance(expr, list):
        return f"{expr[0]}(\n  {', '.join(format_expr(arg) for arg in expr[1:])}\n)".replace("\n  ", "").replace("\n", "")
    return str(expr)

problems = [
    ("p(f(a), g(?Y))", "p(?X, ?X)"),
    ("prime(11)", "prime(?y)"),
    ("Q(a, g(?x, a), f(?y))", "Q(a, g(f(b), a), ?x)"),
    ("p(b, ?X, f(g(?Z)))", "p(?Z, f(?Y), f(?Y))")
]
print("=== ?PREFIX VARIABLE UNIFICATION ===\n")
for i, (s1, s2) in enumerate(problems, 1):
    e1 = parse_expression(s1)
    e2 = parse_expression(s2)

    result = unify(e1, e2)

    print(f"Problem {i}:")
    print(f"  Expr 1: {s1}")
    print(f"  Expr 2: {s2}")

    if result is None:
        print("  Result: ❌ Unification Fails")
    else:
        print("  Result: ✅ Unification Succeeds")
        formatted_mgu = {k: format_expr(v) for k, v in result.items()}
        print(f"  MGU (θ): {formatted_mgu}")
        print(f"  Unified Form: {format_expr(apply_substitution(result, e1))}")
    print("-" * 50)


=== ?PREFIX VARIABLE UNIFICATION ===

Problem 1:
  Expr 1: p(f(a), g(?Y))
  Expr 2: p(?X, ?X)
  Result: ❌ Unification Fails
--------------------------------------------------
Problem 2:
  Expr 1: prime(11)
  Expr 2: prime(?y)
  Result: ✅ Unification Succeeds
  MGU (θ): {'?y': '11'}
  Unified Form: prime(11, ))
--------------------------------------------------
Problem 3:
  Expr 1: Q(a, g(?x, a), f(?y))
  Expr 2: Q(a, g(f(b), a), ?x)
  Result: ❌ Unification Fails
--------------------------------------------------
Problem 4:
  Expr 1: p(b, ?X, f(g(?Z)))
  Expr 2: p(?Z, f(?Y), f(?Y))
  Result: ❌ Unification Fails
--------------------------------------------------


In [ ]:
def unification(delta1,delta2):
  if(delta1==delta2):
    return NULL
  else if("?"in delta1 )